# UniAssist — Week 2: Tokenization, Sentence Boundaries and Subwords
Compare whitespace, NLTK and spaCy on the original corpus. Complete manual tokenization before judging which tokenizer is best.


In [ ]:
from pathlib import Path
import json, re, statistics
from collections import Counter
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "corpus").exists():
    ROOT = ROOT.parent
assert (ROOT / "corpus").exists(), "Run notebook from UniAssist_UMT directory"
metadata = json.loads((ROOT / "metadata.json").read_text(encoding="utf-8"))
records = metadata["documents"]
texts = [(ROOT / "corpus" / row["file"]).read_text(encoding="utf-8") for row in records]
print(len(texts), "documents loaded")


In [ ]:
import nltk, spacy
from nltk.tokenize import word_tokenize, sent_tokenize
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")
methods = {
    "split": lambda t: t.split(),
    "nltk": lambda t: word_tokenize(t, preserve_line=True),
    "spaCy": lambda t: [w.text for w in nlp.make_doc(t) if not w.is_space],
}


## Task A — Token/type counts and type-token ratios


In [ ]:
rows = []
for name, fn in methods.items():
    toks = [word for text in texts for word in fn(text)]
    types = len(set(w.lower() for w in toks))
    punct = sum(not any(ch.isalnum() for ch in word) for word in toks)
    rows.append({"Tokenizer": name, "Tokens": len(toks), "Types": types,
                 "Types/Tokens": round(types / max(1, len(toks)), 4),
                 "Punct. tokens": punct})
display(pd.DataFrame(rows))


## Task B — Regex problem hunt: URLs, emails, dates and punctuation


In [ ]:
patterns = {
    "url": r"https?://\S+|www\.\S+",
    "email": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "number": r"\d[\d.,:/-]*\d|\d",
    "hyphenated": r"\w+(?:-\w+)+",
    "apostrophe": r"\w+['’]\w+",
    "abbreviation": r"\b[A-Z][a-z]{0,3}\.",
    "other_symbol": r"[^\w\s.,;:!?()'\-]"
}
found_rows = []
for kind, pattern in patterns.items():
    matches = [(record["file"], m.group()) for record, text in zip(records, texts)
               for m in re.finditer(pattern, text)]
    found_rows.append({"type": kind, "number": len(matches), "three_examples": matches[:3]})
display(pd.DataFrame(found_rows))


## Task C — Compare the same five tricky expressions across tokenizers


In [ ]:
hard = ["2026–2027", "Erasmus+", "Engineer-Architect",
        "State Matura", "https://umt.edu.al/en/admissions/"]
for sample in hard:
    print("\nEXPRESSION:", sample)
    for name, tokenize in methods.items():
        print(name, len(tokenize(sample)), tokenize(sample))


## Task D — Sentence segmentation and mean/median sentence lengths


In [ ]:
def spacy_sents(text):
    return [s.text.strip() for s in nlp(text).sents if s.text.strip()]

sentences = [s for text in texts for s in spacy_sents(text)]
lengths = [(len(methods["spaCy"](s)), s) for s in sentences]
nums = [n for n,_ in lengths]
print("Sentences", len(sentences), "Mean", statistics.mean(nums), "Median", statistics.median(nums))
for count, sentence in sorted(lengths, reverse=True)[:5]:
    print(count, sentence[:170])
sample = texts[14]
print("Comparing NLTK and spaCy for one document:")
print("NLTK", len(sent_tokenize(sample)), "spaCy", len(spacy_sents(sample)))


## Optional Task E — Subword comparison
If you install transformers and sentencepiece, the following cell downloads model tokenizers. It may require internet access.


In [ ]:
try:
    from transformers import AutoTokenizer
except ImportError:
    print("Optional dependency not installed: pip install transformers sentencepiece")
else:
    models = {"BERT":"bert-base-uncased", "GPT-2":"gpt2",
              "XLM-R":"xlm-roberta-base"}
    for name, checkpoint in models.items():
        try:
            tok = AutoTokenizer.from_pretrained(checkpoint)
            print(name, {word:tok.tokenize(word) for word in
                  ("university", "registration", "Erasmus+", "Metropolitan Tirana")})
        except Exception as err:
            print(name, "could not download:", type(err).__name__)


## Submit
Show the three tokenization tables, five difficult strings, mean/median sentence lengths, five longest sentences and your explanations. Explicitly distinguish manual decisions from software output.
